# Recall@50

전체 30개 질문을 읽고 dev 15개로 설정을 검토합니다.

## 실행 환경 준비

이 노트북은 제공 PDF에서 실제 텍스트를 추출합니다. 답변 생성 API는 호출하지 않습니다.

### Recall 함수의 입력과 경계

`search()`가 같은 문서의 여러 쪽 가운데 최고 점수 한 쪽만 남긴 고유 문서 순위를 반환합니다. `recall_at()`은 이 결과의 앞 `k`개와 정답 집합을 비교합니다. 중복 제거는 `recall_at()`이 아니라 `search()`의 책임입니다. 반환값은 질문 한 건의 점수이고 화면의 평균은 질문별 값을 다시 평균한 것입니다. 정답이 6위면 Recall@5에는 없고 Recall@50에는 있으므로 큰 k의 높은 점수는 첫 화면 순서를 설명하지 않습니다.

In [1]:
from pathlib import Path
import json, math, re
from collections import Counter
from pypdf import PdfReader

# 현재 폴더에서 저장소의 공개 RAG 자료를 찾습니다.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "labs/data/rag-corpus/manifest.json").exists())
corpus = root / "labs/data/rag-corpus"
dataset_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
dataset = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines()]
evaluation_items = [item for item in dataset if item["split"] == "dev"]
print(f"전체 질문: {len(dataset)}개")
print(f"설정 선택용 dev 질문: {len(evaluation_items)}개")

전체 질문: 30개
설정 선택용 dev 질문: 15개


## 공개 PDF를 페이지 단위 후보로 준비

스캔본은 OCR이 필요하므로 이번 기초 검색에서 제외합니다.

In [2]:
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
pages = []
for item in manifest:
    if item["tier"] == "scanned":
        continue
    reader = PdfReader(corpus / item["file"])
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"doc_id": item["doc_id"], "page": page_no, "text": text})
print(f"검색 후보: {len(pages)}개 페이지")

검색 후보: 90개 페이지


## 같은 검색기를 모든 질문에 적용

단어와 한글 두 글자 조각의 겹침을 사용합니다. 점수가 같으면 문서 ID와 쪽 번호로 순서를 고정합니다.

In [3]:
def tokens(text):
    normalized = re.sub(r"\s+", " ", text.lower())
    words = re.findall(r"[가-힣a-z0-9]+", normalized)
    compact = re.sub(r"[^가-힣a-z0-9]", "", normalized)
    return words + [compact[i:i+2] for i in range(max(0, len(compact)-1))]

def search(question, limit=50):
    query_counts = Counter(tokens(question))
    scored = []
    for page in pages:
        page_counts = Counter(tokens(page["text"]))
        overlap = sum(min(count, page_counts[token]) for token, count in query_counts.items())
        score = overlap / math.sqrt(max(1, sum(page_counts.values())))
        scored.append({"doc_id": page["doc_id"], "page": page["page"], "score": score})
    ranked_pages = sorted(scored, key=lambda row: (-row["score"], row["doc_id"], row["page"]))
    unique_docs, seen = [], set()
    for row in ranked_pages:
        if row["doc_id"] not in seen:
            unique_docs.append(row)
            seen.add(row["doc_id"])
    return unique_docs[:limit]

sample = search(dataset[0]["question"], 5)
print([(row["doc_id"], row["page"], round(row["score"], 3)) for row in sample])

[('E01', 1, 0.926), ('M04', 3, 0.225), ('H03', 1, 0.188), ('H06', 5, 0.142), ('E02', 5, 0.132)]


## Recall@50 계산

정답 문서 중 후보 50개 안에 든 문서의 비율을 이 문항의 재현율로 계산합니다.

In [4]:
def recall_at(results, gold, k):
    found = {row["doc_id"] for row in results[:k]}
    return len(found & set(gold)) / len(gold) if gold else 0.0

rows = []
for item in evaluation_items:
    results = search(item["question"], 50)
    rows.append((item["qid"], recall_at(results, item["gold_doc_ids"], 50)))
print("Recall@50:", round(sum(score for _, score in rows) / len(rows), 3))
print("놓친 질문:", [qid for qid, score in rows if score < 1])

Recall@50: 1.0
놓친 질문: []


## k 값 비교

k만 바꾸고 질문·라벨·검색 점수를 고정합니다.

In [5]:
for k in (1, 5, 10, 50):
    scores = [recall_at(search(item["question"], 50), item["gold_doc_ids"], k)
              for item in evaluation_items]
    print(f"Recall@{k}: {sum(scores) / len(scores):.3f}")

Recall@1: 0.733


Recall@5: 1.000


Recall@10: 1.000


Recall@50: 1.000


## 51위 누락을 작은 목록으로 확인

현재 말뭉치는 검색 가능한 고유 문서가 17개라 Recall@50이 포화됩니다. 51개 목록으로 경계를 따로 검증합니다.

In [6]:
toy = [{"doc_id": f"D{i:02d}"} for i in range(1, 52)]
assert recall_at(toy, ["D50"], 50) == 1.0
assert recall_at(toy, ["D51"], 50) == 0.0
print("정답 50위:", recall_at(toy, ["D50"], 50))
print("정답 51위:", recall_at(toy, ["D51"], 50))

정답 50위: 1.0
정답 51위: 0.0
